# PRO134 – HMI interactivo (Normal + Emergencia)

Ejecuta la celda siguiente para iniciar el HMI.

In [1]:

import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

# ============================================================
# PRO134 – Consumo de materiales y repuestos
# HMI oficial (base estilo PRO130 / PRO134 aprobado)
#
# Inicio (selector):
#   - Consumo en condiciones normales
#   - Consumo bajo emergencia
#
# Condiciones normales:
#   - 2 inicios operativos (Solicitante / Almacén)
#   - Convergencia en "Picking realizado"
#
# Emergencia:
#   - Flujo de "Consumo de materiales bajo emergencia"
#   - Convergencia post-rombo en "Registrar salida de material..."
#
# Reglas HMI:
# - Títulos de pasos = casillas del flujo (texto exacto)  [excepto selectores HMI]
# - Checklist por paso (no avanza si no se completa)
# - Botón NO => BLOQUEADO con motivo obligatorio + "Rehacer paso"
# - Volver al paso anterior
# - Exporta JSON auditable (run_id, historial, decisiones, bloqueos, timestamps, estados)
# ============================================================

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

# Motivos de bloqueo (multi-selección) - base PRO134 (aplica a ambos modos)
MOTIVOS_BLOQUEO_PRO134 = [
    "Stock insuficiente / quiebre",
    "Diferencia stock físico vs sistema",
    "Material no apto (vencido / deteriorado / embalaje dañado)",
    "Reserva incompleta / inconsistente",
    "No es posible registrar en sistema (SAP indisponible / sin acceso)",
    "Falta información mínima para enriquecer (cuando aplica)",
    "Otro"
]

# -------------------------
# NODOS
# -------------------------
NODOS = {
    # ===== Selector principal (HMI) =====
    "S0_modo": {
        "type": "decision",
        "titulo": "¿El consumo es bajo emergencia o en condiciones normales?",
        "rol": "HMI (selección de modo)",
        "descripcion": "Seleccione el modo operativo de consumo según el contexto en terreno.",
        "pregunta": "Seleccione una opción para iniciar:",
        "opciones": [
            {"label": "Consumo en condiciones normales", "next": "N0_inicio"},
            {"label": "Consumo bajo emergencia", "next": "E1_informar"},
        ],
        "ayuda": "Este selector define el flujo a ejecutar, sin modificar los pasos del PRO."
    },

    # ===== Consumo normal: selector de inicio operativo (HMI) =====
    "N0_inicio": {
        "type": "decision",
        "titulo": "¿Cómo se inicia el consumo de materiales?",
        "rol": "HMI (selección de inicio)",
        "descripcion": "Seleccione el punto de entrada operativo según el contexto en terreno.",
        "pregunta": "Seleccione una opción para iniciar:",
        "opciones": [
            {"label": "Opción 1: Necesidad de consumo (Solicitante)", "next": "A1_necesidad"},
            {"label": "Opción 2: Revisión diaria de reservas (Almacén)", "next": "B1_consultar_reservas"},
        ],
        "ayuda": "Este selector define el punto de entrada, sin modificar el flujo del PRO."
    },

    # ---- Flujo A (Solicitante) ----
    "A1_necesidad": {
        "type": "task",
        "titulo": "Necesidad de consumo de materiales/EPPs",
        "rol": "Solicitante de material/repuesto",
        "descripcion": "Se identifica en terreno la necesidad de consumo de materiales o EPPs para ejecutar el trabajo.",
        "acciones": [
            "Confirmar material/EPP requerido y cantidad.",
            "Definir contexto de uso (OT / consumo manual según aplique)."
        ],
        "checklist": [
            "Material/EPP identificado",
            "Cantidad definida",
            "Contexto de uso claro (OT / consumo manual)"
        ],
        "validacion": "¿La necesidad está claramente definida (material/EPP + cantidad) y corresponde al contexto del trabajo?",
        "next": "A2_generar_reserva"
    },
    "A2_generar_reserva": {
        "type": "task",
        "titulo": "Generar reserva materiales",
        "rol": "Solicitante de material/repuesto",
        "descripcion": "Generar la reserva de materiales en el sistema para habilitar el picking y entrega.",
        "acciones": [
            "Generar reserva en SAP (MB21).",
            "Ingresar fecha de necesidad y datos requeridos.",
            "Confirmar número de reserva."
        ],
        "checklist": [
            "Reserva creada en SAP (MB21)",
            "Fecha de necesidad ingresada",
            "Número de reserva disponible"
        ],
        "validacion": "¿La reserva quedó generada en SAP y se dispone del número de reserva?",
        "next": "A3_stock_suficiente"
    },
    "A3_stock_suficiente": {
        "type": "decision",
        "titulo": "¿Tiene stock suficiente?",
        "rol": "Solicitante de material/repuesto",
        "descripcion": "Validar disponibilidad para cumplir la fecha de necesidad de la reserva.",
        "pregunta": "¿Existe stock para cumplir la fecha de necesidad definida en la reserva?",
        "opciones": [
            {"label": "SÍ", "next": "A4_fecha_necesidad"},
            {"label": "NO", "next": "C1_compra"},
        ],
    },
    "A4_fecha_necesidad": {
        "type": "task",
        "titulo": "Fecha de necesidad cumplida",
        "rol": "Solicitante de material/repuesto",
        "descripcion": "Se confirma que la fecha de necesidad se cumple. Desde aquí el flujo converge al tramo común.",
        "acciones": ["Coordinar retiro/recepción con almacén según corresponda."],
        "checklist": ["Fecha de necesidad confirmada", "Coordinación con almacén considerada"],
        "validacion": "¿Se confirma que la fecha de necesidad será cumplida y el material podrá ser retirado/entregado?",
        "next": "COMUN_0_picking_realizado"
    },

    # ---- Flujo B (Almacén) ----
    "B1_consultar_reservas": {
        "type": "task",
        "titulo": "Consultar reservas de materiales para picking",
        "rol": "Especialista de almacén",
        "descripcion": "Revisión diaria de reservas para planificar y ejecutar el picking.",
        "acciones": [
            "Consultar reservas en SAP (MB25).",
            "Identificar reservas a preparar en el día/turno."
        ],
        "checklist": ["Reservas consultadas en SAP (MB25)", "Reservas del día identificadas"],
        "validacion": "¿Se revisaron las reservas y se identificaron las que requieren picking hoy?",
        "next": "B2_stock_suficiente"
    },
    "B2_stock_suficiente": {
        "type": "decision",
        "titulo": "¿Tiene stock suficiente?",
        "rol": "Especialista de almacén",
        "descripcion": "Validar disponibilidad de stock para ejecutar el picking de la(s) reserva(s) identificada(s).",
        "pregunta": "¿Existe stock disponible hoy para ejecutar el picking de la(s) reserva(s) identificada(s)?",
        "opciones": [
            {"label": "SÍ", "next": "B3_realizar_picking"},
            {"label": "NO", "next": "C1_compra"},
        ],
    },
    "B3_realizar_picking": {
        "type": "task",
        "titulo": "Realizar picking de materiales",
        "rol": "Especialista de almacén",
        "descripcion": "Ejecutar el picking de materiales según la reserva.",
        "acciones": [
            "Retirar material según la reserva.",
            "Verificar condición del material (apto/no apto) antes de entregar."
        ],
        "checklist": [
            "Material retirado según reserva",
            "Condición verificada (no vencido/deteriorado)",
            "Material preparado para entrega"
        ],
        "validacion": "¿El picking fue realizado y el material está en condición apta para entrega?",
        "next": "COMUN_0_picking_realizado"
    },

    # ---- Rama compra/recepción/notificación (convergente) ----
    "C1_compra": {
        "type": "task",
        "titulo": "Compra de materiales y repuesto",
        "rol": "MRP",
        "descripcion": "Gestión de compra de materiales/repuestos cuando no existe stock suficiente.",
        "acciones": ["Iniciar proceso de compra según estándar MRP."],
        "checklist": ["Compra iniciada por MRP"],
        "validacion": "¿Se inició la compra de materiales/repuestos por MRP?",
        "next": "C2_recepcion"
    },
    "C2_recepcion": {
        "type": "task",
        "titulo": "Recepción de material",
        "rol": "Especialista de almacén",
        "descripcion": "Recepcionar material comprado para continuar con el proceso.",
        "acciones": [
            "Recepcionar material en almacén.",
            "Dejar material disponible para retiro/entrega."
        ],
        "checklist": ["Material recepcionado", "Material disponible para retiro/entrega"],
        "validacion": "¿El material fue recepcionado y quedó disponible para el solicitante?",
        "next": "C3_notificar"
    },
    "C3_notificar": {
        "type": "task",
        "titulo": "Notificar a solicitante llegada de material",
        "rol": "Especialista de almacén",
        "descripcion": "Notificar al solicitante que el material llegó para coordinar entrega/retiro.",
        "acciones": ["Notificar llegada de material al solicitante por canal definido."],
        "checklist": ["Solicitante notificado"],
        "validacion": "¿Se notificó efectivamente al solicitante la llegada del material?",
        "next": "COMUN_0_picking_realizado"
    },

    # ---- Tramo común (normal) ----
    "COMUN_0_picking_realizado": {
        "type": "task",
        "titulo": "Picking realizado",
        "rol": "Especialista de almacén",
        "descripcion": "Hito de convergencia: el picking ya fue ejecutado (o el material ya está disponible).",
        "acciones": ["Continuar con verificación, entrega y registro de salida."],
        "checklist": ["Material disponible para verificación/entrega"],
        "validacion": "¿El material está disponible para continuar con la verificación y entrega?",
        "next": "COMUN_1_verificar_reserva"
    },
    "COMUN_1_verificar_reserva": {
        "type": "task",
        "titulo": "Verificar reserva de materiales",
        "rol": "Especialista de almacén",
        "descripcion": "Verificar que lo preparado/entregado coincide con lo indicado en la reserva.",
        "acciones": [
            "Comparar materiales/cantidades vs reserva.",
            "Corregir discrepancias antes de entregar."
        ],
        "checklist": ["Material coincide con reserva", "Cantidad coincide con reserva"],
        "validacion": "¿Los materiales y cantidades coinciden con la reserva?",
        "next": "COMUN_2_entregar_materiales"
    },
    "COMUN_2_entregar_materiales": {
        "type": "task",
        "titulo": "Entregar materiales",
        "rol": "Especialista de almacén",
        "descripcion": "Entregar materiales al solicitante con control correspondiente.",
        "acciones": [
            "Entregar materiales al solicitante.",
            "Asegurar control para evitar pérdida/hurto."
        ],
        "checklist": ["Material entregado", "Entrega bajo control (documento/vale)"],
        "validacion": "¿Los materiales fueron entregados al solicitante bajo control (documento/vale)?",
        "next": "COMUN_3_registrar_salida_enriquecer"
    },
    "COMUN_3_registrar_salida_enriquecer": {
        "type": "task",
        "titulo": "Registrar salida de material y enriquecer material en caso de contar con información",
        "rol": "Especialista de almacén",
        "descripcion": "Registrar la salida en sistema y, si se cuenta con información, enriquecer el material.",
        "acciones": [
            "Registrar salida de material en SAP (MIGO / MB1A según corresponda).",
            "Si se cuenta con información: completar enriquecimiento del material (clasificación/datos técnicos/foto/texto breve/ubicación técnica/stock de seguridad/criticidad)."
        ],
        "checklist": [
            "Salida registrada en sistema (SAP)",
            "Si aplica: clasificación definida",
            "Si aplica: datos técnicos completados",
            "Si aplica: foto incorporada",
            "Si aplica: texto breve estándar",
            "Si aplica: ubicación técnica vinculada",
            "Si aplica: stock de seguridad/criticidad definidos"
        ],
        "validacion": "¿La salida quedó registrada en el sistema y (si aplica) se completó el enriquecimiento disponible?",
        "next": "COMUN_4_firma_solicitante"
    },
    "COMUN_4_firma_solicitante": {
        "type": "task",
        "titulo": "Firmar Documento de Reserva y Vale de Acompañamiento",
        "rol": "Solicitante de material/repuesto",
        "descripcion": "Firma del solicitante para trazabilidad del retiro/entrega de material.",
        "acciones": ["Firmar Documento de Reserva y Vale de Acompañamiento."],
        "checklist": ["Firma solicitante realizada"],
        "inputs": [
            {"key": "documento_firma_normal", "label": "Ingresa documento", "required": True}
        ],
        "validacion": "¿El solicitante firmó el Documento de Reserva y Vale de Acompañamiento?",
        "next": "COMUN_5_firma_almacen"
    },
    "COMUN_5_firma_almacen": {
        "type": "task",
        "titulo": "Firmar doc. reserva",
        "rol": "Especialista de almacén",
        "descripcion": "Firma del especialista de almacén para cierre del movimiento.",
        "acciones": ["Firmar documento de reserva."],
        "checklist": ["Firma almacén realizada"],
        "validacion": "¿El especialista de almacén firmó el documento de reserva?",
        "next": "COMUN_6_archivar"
    },
    "COMUN_6_archivar": {
        "type": "task",
        "titulo": "Archivar doc. reserva",
        "rol": "Especialista de almacén",
        "descripcion": "Archivar la documentación según estándar local para auditoría y trazabilidad.",
        "acciones": ["Archivar documento de reserva según estándar del almacén."],
        "checklist": ["Documento archivado"],
        "validacion": "¿El documento fue archivado correctamente?",
        "next": "END_NORMAL"
    },

    "END_NORMAL": {
        "type": "end",
        "titulo": "🏁 Proceso finalizado",
        "rol": "HMI",
        "descripcion": "Se completaron los pasos del PRO134 (condiciones normales). Puede exportar el JSON auditable si lo requiere.",
        "mensaje": "Ya está en el final/cierre del procedimiento.",
        "estado_final": FINALIZADO
    },

    # ===== Emergencia =====
    "E1_informar": {
        "type": "task",
        "titulo": "Informar anomalía que requiere atención",
        "rol": "Solicitante",
        "descripcion": "Se informa la anomalía detectada que requiere atención inmediata.",
        "acciones": ["Informar la anomalía y el contexto operativo asociado."],
        "checklist": ["Anomalía informada"],
        "validacion": "¿La anomalía fue informada con información suficiente para su evaluación?",
        "next": "E2_validar_emergencia"
    },
    "E2_validar_emergencia": {
        "type": "task",
        "titulo": "Validar emergencia",
        "rol": "Jefe de turno",
        "descripcion": "Se valida si la situación corresponde a una emergencia según el criterio del procedimiento.",
        "acciones": ["Evaluar si la situación califica como emergencia."],
        "checklist": ["Emergencia validada"],
        "validacion": "¿Se validó que la situación corresponde a una emergencia?",
        "next": "E3_notificar_almacen"
    },
    "E3_notificar_almacen": {
        "type": "task",
        "titulo": "Notificar emergencia a Responsable de Almacén",
        "rol": "Jefe de turno",
        "descripcion": "Se notifica la emergencia al responsable de almacén para habilitar el retiro urgente.",
        "acciones": ["Notificar emergencia al Responsable de Almacén."],
        "checklist": ["Responsable de Almacén notificado"],
        "validacion": "¿Se notificó la emergencia al Responsable de Almacén?",
        "next": "E4_formulario_retiro"
    },
    "E4_formulario_retiro": {
        "type": "task",
        "titulo": "Completar Formulario de Retiro y retirar material del almacén",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Se completa el formulario de retiro bajo emergencia y se retira material del almacén.",
        "acciones": ["Completar Formulario de Retiro y retirar el material requerido del almacén."],
        "checklist": ["Formulario de Retiro completado", "Material retirado del almacén"],
        "validacion": "¿Formulario completado y material retirado del almacén?",
        "next": "E5_actividades"
    },
    "E5_actividades": {
        "type": "task",
        "titulo": "Realizar actividades para abordar emergencia",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Se ejecutan las actividades necesarias para abordar la emergencia.",
        "acciones": ["Realizar actividades para abordar la emergencia."],
        "checklist": ["Actividades ejecutadas para abordar emergencia"],
        "validacion": "¿Se realizaron las actividades para abordar la emergencia?",
        "next": "E6_equipo_activo"
    },
    "E6_equipo_activo": {
        "type": "decision",
        "titulo": "¿La emergencia se basa en una falla en un equipo activo?",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Definir el mecanismo de regularización: vía OT (equipo activo) o reserva manual (no equipo activo).",
        "pregunta": "¿La emergencia se basa en una falla en un equipo activo?",
        "opciones": [
            {"label": "SÍ", "next": "E7_incorporar_ot"},
            {"label": "NO", "next": "E7N_reserva_manual"},
        ],
    },
    "E7_incorporar_ot": {
        "type": "task",
        "titulo": "Incorporar materiales en OT",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Se incorporan los materiales utilizados en la OT correspondiente.",
        "acciones": ["Incorporar materiales en OT."],
        "checklist": ["Materiales incorporados en OT"],
        "validacion": "¿Los materiales fueron incorporados en la OT?",
        "next": "E8_imprimir_ot"
    },
    "E8_imprimir_ot": {
        "type": "task",
        "titulo": "Imprimir reserva desde OT",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Se imprime la reserva desde la OT para respaldar el consumo bajo emergencia.",
        "acciones": ["Imprimir reserva desde OT."],
        "checklist": ["Reserva impresa desde OT"],
        "inputs": [
            {"key": "numero_reserva_ot", "label": "Ingresa numero de reserva", "required": True}
        ],
        "validacion": "¿Se imprimió la reserva desde OT?",
        "next": "E9_registrar_salida"
    },
    "E7N_reserva_manual": {
        "type": "task",
        "titulo": "Generar reserva manual",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Se genera reserva manual para respaldar el consumo bajo emergencia cuando no corresponde a equipo activo.",
        "acciones": ["Generar reserva manual."],
        "checklist": ["Reserva manual generada"],
        "inputs": [
            {"key": "numero_reserva_manual", "label": "Ingresa numero de reserva", "required": True}
        ],
        "validacion": "¿Se generó la reserva manual?",
        "next": "E9_registrar_salida"
    },

    # Convergencia emergencia
    "E9_registrar_salida": {
        "type": "task",
        "titulo": "Registrar salida de material y enriquecer material en caso de contar con información",
        "rol": "Especialista de almacén",
        "descripcion": "Registrar la salida en sistema y, si se cuenta con información, enriquecer el material.",
        "acciones": [
            "Registrar salida de material en el sistema.",
            "Si se cuenta con información: completar enriquecimiento del material."
        ],
        "checklist": [
            "Salida registrada en sistema",
            "Si aplica: clasificación definida",
            "Si aplica: datos técnicos completados",
            "Si aplica: foto incorporada",
            "Si aplica: texto breve estándar",
            "Si aplica: ubicación técnica vinculada",
            "Si aplica: stock de seguridad/criticidad definidos"
        ],
        "validacion": "¿La salida quedó registrada en el sistema y (si aplica) se completó el enriquecimiento disponible?",
        "next": "E10_firmar_regularizar"
    },
    "E10_firmar_regularizar": {
        "type": "task",
        "titulo": "Firmar Documento de Reserva y regularizar Formulario de retiro de emergencia",
        "rol": "Supervisión de mant./ Supervisión de operaciones/ solicitante",
        "descripcion": "Se firma el documento de reserva y se regulariza el formulario de retiro bajo emergencia.",
        "acciones": ["Firmar Documento de Reserva y regularizar Formulario de retiro de emergencia."],
        "checklist": ["Documento de Reserva firmado (regularizacion)", "Formulario de retiro de emergencia regularizado"],
        "inputs": [
            {"key": "documento_firma_emergencia", "label": "Ingresa documento", "required": True}
        ],
        "validacion": "¿Se firmo el documento y se regularizo el formulario de retiro de emergencia?",
        "next": "E11_firma_almacen"
    },
    "E11_firma_almacen": {
        "type": "task",
        "titulo": "Firmar Documento de Reserva",
        "rol": "Especialista de almacén",
        "descripcion": "Firma del especialista de almacén para cierre del documento de reserva.",
        "acciones": ["Firmar Documento de Reserva."],
        "checklist": ["Documento de Reserva firmado por almacén"],
        "validacion": "¿El especialista de almacén firmó el Documento de Reserva?",
        "next": "E12_archivar"
    },
    "E12_archivar": {
        "type": "task",
        "titulo": "Archivar Documento de reserva",
        "rol": "Especialista de almacén",
        "descripcion": "Archivar el documento de reserva asociado al retiro bajo emergencia.",
        "acciones": ["Archivar Documento de reserva."],
        "checklist": ["Documento de reserva archivado"],
        "validacion": "¿El Documento de reserva fue archivado?",
        "next": "END_EMERGENCIA"
    },

    "END_EMERGENCIA": {
        "type": "end",
        "titulo": "🏁 Proceso finalizado",
        "rol": "HMI",
        "descripcion": "Se completaron los pasos del PRO134 (consumo bajo emergencia). Puede exportar el JSON auditable si lo requiere.",
        "mensaje": "Ya está en el final/cierre del procedimiento.",
        "estado_final": FINALIZADO
    }
}

# -------------------------
# HMI (estilo visual PRO130 / PRO134 aprobado)
# -------------------------
class PRO134HMI:
    def __init__(self):
        self.nodo_id = "S0_modo"
        self.historial = []
        self.logs = []
        self.decisiones = []
        self.bloqueos = []

        self.run_id = str(uuid.uuid4())
        self.estado = EN_CURSO
        self.start_ts = _now_iso()
        self.end_ts = None

        # Metadata útil (sin afectar el flujo)
        self.modo = None  # "NORMAL" / "EMERGENCIA" (se captura desde el selector)

        self.output = widgets.Output(layout={"width":"100%"})

        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])

        self.is_blocked = False
        self.block_panel = widgets.VBox([])
        self.btn_rehacer = widgets.Button(description="🔄 Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._decision_widget = None
        self._check_widgets = []
        self._input_widgets = {}
        self.input_values = {}

        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "estado": self.estado,
            "nodo": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self):
        self.historial.append(self.nodo_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    def _render_header(self, n):
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;color:black;'><b>ROL:</b> {n.get('rol','')}</span>"
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO134</b> – Consumo de materiales y repuestos</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        acciones = "".join([f"<li style='margin:4px 0;color:#0f172a;'>{a}</li>" for a in n.get("acciones",[])])
        valid = n.get("validacion","")

        self._check_widgets = [widgets.Checkbox(description=item, value=False) for item in (n.get("checklist",[]) or [])]
        self._input_widgets = {}

        accion_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>ACCION A EJECUTAR (texto PRO134)</b></div>
                <ul style="margin-top:10px;padding-left:18px;color:#0f172a;">{acciones}</ul>
            </div>
        """)

        checklist_box = widgets.VBox([])
        if self._check_widgets:
            checklist_items = []
            for cb in self._check_widgets:
                label = (cb.description or "").strip().lower()
                obligatorio = "si aplica" not in label
                req = "Obligatorio" if obligatorio else "No obligatorio"
                row = widgets.HBox([
                    cb,
                    widgets.HTML(f"<span style='font-size:11px;color:#475569;'>{req}</span>")
                ])
                checklist_items.append(row)

            checklist_box = widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>CHECKLIST</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Marca cada item al completar en terreno. Los que dicen 'si aplica' no bloquean el avance.</div>
                </div>
                """),
                widgets.VBox(checklist_items)
            ])

        input_box = widgets.VBox([])
        input_widgets = []
        for cfg in (n.get("inputs", []) or []):
            key = cfg.get("key")
            label = cfg.get("label", key or "Campo")
            required = cfg.get("required", False)
            widget = widgets.Text(
                value=self.input_values.get(self.nodo_id, {}).get(key, ""),
                placeholder=label,
                layout=widgets.Layout(width="100%")
            )
            self._input_widgets[key] = {"widget": widget, "config": cfg}
            req = "Obligatorio" if required else "Opcional"
            input_widgets.append(widgets.VBox([
                widgets.HTML(f"<div style='font-size:12px;color:#0f172a;margin-top:6px;'><b>{label}</b> <span style='color:#475569;'>({req})</span></div>"),
                widget
            ]))

        if input_widgets:
            input_box = widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>REGISTRO DE DATOS</b></div>
                </div>
                """),
                widgets.VBox(input_widgets)
            ])

        valid_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>VALIDACION</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SI</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
            </div>
        """)

        return widgets.VBox([accion_box, checklist_box, input_box, valid_box])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ]), radios

    def _render_block_panel(self):
        if not self.is_blocked:
            self.block_panel.children = []
            return

        title = widgets.HTML("""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #ef4444;background:#fff1f2;">
            <div style="font-size:14px;color:#0f172a;"><b>⛔ BLOQUEADO</b> — Seleccione motivo(s) y registre detalle.</div>
            <div style="margin-top:8px;font-size:12px;color:#0f172a;">No puede avanzar hasta rehacer el paso.</div>
        </div>
        """)

        self.sel_motivos = widgets.SelectMultiple(options=MOTIVOS_BLOQUEO_PRO134, rows=7, layout={"width":"100%"})
        self.txt_detalle = widgets.Textarea(
            placeholder="Detalle del bloqueo (obligatorio si selecciona 'Otro').",
            layout=widgets.Layout(width="100%", height="80px")
        )

        self.block_panel.children = [
            title,
            widgets.HTML("<b>Motivo(s) de bloqueo:</b> (selección múltiple)"),
            self.sel_motivos,
            widgets.HTML("<b>Detalle:</b>"),
            self.txt_detalle,
            self.btn_rehacer
        ]

    def _render_footer(self):
        self.btn_volver.disabled = (len(self.historial) == 0)
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()
            self._clear_msg()

            n = NODOS[self.nodo_id]
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
                self._check_widgets = []
                self._input_widgets = {}
            elif n["type"] == "end":
                self._decision_widget = None
                self._check_widgets = []
                self._input_widgets = {}
                body = widgets.HTML(f"""
                <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
                    <div style="font-size:20px;color:#0f172a;"><b>🏁 FIN</b></div>
                    <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','')}</div>
                    <div style="margin-top:10px;font-size:12px;color:#0f172a;">Estado final: <b>{n.get('estado_final','')}</b></div>
                </div>
                """)
            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")
                self._decision_widget = None
                self._check_widgets = []

            self._render_block_panel()
            footer = self._render_footer()
            self.main_box.children = [header, body, footer]
            display(self.main_box)

    def _check_ready_to_advance(self):
        n = NODOS[self.nodo_id]

        if self.is_blocked:
            return False, "Paso bloqueado. Registre motivo(s) y use 'Rehacer paso'."

        if n["type"] == "end":
            return True, ""

        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                return False, "Debe seleccionar una opción para avanzar."
            return True, ""

        if n["type"] == "task":
            if self._check_widgets:
                pendientes = []
                for cb in self._check_widgets:
                    label = (cb.description or "").strip().lower()
                    obligatorio = "si aplica" not in label
                    if obligatorio and not cb.value:
                        pendientes.append(cb.description)
                if pendientes:
                    return False, "Debe completar el checklist obligatorio antes de avanzar."

            if self._input_widgets:
                for key, meta in self._input_widgets.items():
                    cfg = meta.get("config", {})
                    value = (meta.get("widget").value or "").strip()
                    if cfg.get("required") and not value:
                        return False, f"Debe completar el campo: {cfg.get('label', key)}."
            return True, ""

        return True, ""

    def _advance_to(self, next_id):
        if next_id not in NODOS:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'>
                <b>⚠ Error de flujo:</b> el nodo destino no existe: <code>{next_id}</code>
            </div>
            """)
            self._log("ERROR_FLUJO", {"missing_next": next_id})
            return
        self.nodo_id = next_id
        self._render()

    def _capture_mode_from_selection(self, node_id, chosen_label):
        # Captura modo desde el selector principal (sin afectar el flujo)
        if node_id == "S0_modo":
            if "emergencia" in (chosen_label or "").lower():
                self.modo = "EMERGENCIA"
            elif "normales" in (chosen_label or "").lower() or "condiciones normales" in (chosen_label or "").lower():
                self.modo = "NORMAL"

    def _on_si(self, _):
        ok, msg = self._check_ready_to_advance()
        if not ok:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ {msg}</b>
            </div>
            """)
            self._log("VALIDACION_FALLA", {"mensaje": msg})
            return

        n = NODOS[self.nodo_id]

        if n["type"] == "end":
            self.estado = FINALIZADO
            self.end_ts = _now_iso()
            self._log("FINALIZA")
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
                <b>🏁 Ya está en el final/cierre.</b>
            </div>
            """)
            return

        self._push_hist()

        if n["type"] == "task":
            if self._input_widgets:
                self.input_values[self.nodo_id] = {
                    key: (meta.get("widget").value or "").strip()
                    for key, meta in self._input_widgets.items()
                }
            self._log("AVANZA", {"next": n.get("next"), "inputs": self.input_values.get(self.nodo_id, {})})
            self._advance_to(n.get("next"))
        elif n["type"] == "decision":
            chosen_next = self._decision_widget.value
            chosen_label = next((o["label"] for o in n.get("opciones",[]) if o["next"] == chosen_next), None)

            self._capture_mode_from_selection(self.nodo_id, chosen_label)

            self.decisiones.append({
                "ts": _now_iso(),
                "nodo": self.nodo_id,
                "titulo": n.get("titulo",""),
                "seleccion": chosen_label,
                "next": chosen_next
            })
            self._log("DECISION", {"seleccion": chosen_label, "next": chosen_next})
            self._advance_to(chosen_next)

    def _on_no(self, _):
        if self.is_blocked:
            return
        self.is_blocked = True
        self.estado = BLOQUEADO
        self.block_ts_inicio = _now_iso()
        self._log("BLOQUEADO_INICIO")
        self._render()

    def _on_rehacer(self, _):
        motivos = list(self.sel_motivos.value) if hasattr(self, "sel_motivos") else []
        detalle = (self.txt_detalle.value or "").strip() if hasattr(self, "txt_detalle") else ""

        if not motivos:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe seleccionar al menos un motivo.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "sin_motivo"})
            return

        if "Otro" in motivos and not detalle:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe ingresar detalle si selecciona 'Otro'.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "otro_sin_detalle"})
            return

        bloqueo = {
            "ts_inicio": getattr(self, "block_ts_inicio", None),
            "ts_fin": _now_iso(),
            "nodo": self.nodo_id,
            "titulo": NODOS[self.nodo_id].get("titulo",""),
            "motivos": motivos,
            "detalle": detalle
        }
        self.bloqueos.append(bloqueo)
        self._log("BLOQUEADO_FIN", bloqueo)

        self.is_blocked = False
        self.estado = EN_CURSO
        self._log("REHACER_PASO")
        self._render()

    def _on_volver(self, _):
        if self.is_blocked:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ No puede volver mientras el paso está bloqueado. Use 'Rehacer paso'.</b>
            </div>
            """)
            return

        prev_id = self._pop_hist()
        if prev_id is not None:
            self._log("VOLVER", {"to": prev_id})
            self._advance_to(prev_id)

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO134 – Consumo de materiales y repuestos",
            "run_id": self.run_id,
            "estado": self.estado,
            "start_ts": self.start_ts,
            "end_ts": self.end_ts,
            "modo": self.modo,
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "decisiones": list(self.decisiones),
            "bloqueos": list(self.bloqueos),
            "inputs": dict(self.input_values),
            "logs": list(self.logs),
            "export_ts": _now_iso(),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(self.output)

hmi = PRO134HMI()
hmi.iniciar()


Output(layout=Layout(width='100%'))